# 09 专家并行与上下文并行

## 简介

本笔记本介绍两种重要的并行策略：
- **专家并行（Expert Parallelism, EP）**: 用于 MoE（Mixture of Experts）模型，将 Expert 分布到不同 GPU
- **上下文并行（Context Parallelism, CP）**: 用于长序列场景，沿序列维度切分注意力计算

这两种策略在大模型时代非常重要——MoE 是 DeepSeek-V3/Mixtral 等模型的核心，
CP 是处理 128K+ 超长上下文的必要手段。

In [ ]:
import torch
import sys; sys.path.insert(0, '..')
from parallel.expert_parallel.expert_partition import partition_experts, get_expert_owner
from parallel.expert_parallel.token_dispatch import dispatch_tokens_to_experts
from parallel.context_parallel.ring_attention import ring_attention_step, rotate_kv
from parallel.context_parallel.sequence_partition import partition_sequence, create_cp_causal_mask
from parallel.context_parallel.cp_integration import analyze_cp_tp_memory, recommend_parallel_config

print("专家并行与上下文并行模块加载完成")

### 🧠 直觉理解：专家并行与上下文并行

**专家并行**就像一个大型咨询公司：公司有 256 位顾问（Expert），但每个客户（token）只需要咨询 2-3 位最匹配的顾问。通过分诊台（Router）把客户引导到对应顾问所在的办公室（GPU），咨询完再把结果送回。

**上下文并行**就像多人合作阅读一本超长小说：每人只读其中一章，但通过环形传阅（Ring Attention），每个人最终都能了解所有章节的内容，同时每人只需要记住自己那一章的笔记（KV Cache）。

### 📐 数学原理：Ring Attention

Ring Attention 将序列 $S$ 均分到 $P$ 个 GPU，每个 GPU 处理 $S/P$ 个 token。

对于 GPU $k$，在第 $s$ 步持有 $Q_k$ 和旋转后的 $KV_{(k+s) \mod P}$：

$$O_k^{(s)} = \text{FlashAttn}(Q_k, K_{(k+s)\%P}, V_{(k+s)\%P})$$

使用在线 softmax 累积更新：

$$m_{new} = \max(m_{old}, m_{partial}), \quad l_{new} = e^{m_{old}-m_{new}} l_{old} + e^{m_{partial}-m_{new}} l_{partial}$$

$$O_{new} = \frac{e^{m_{old}-m_{new}} l_{old} \cdot O_{old} + e^{m_{partial}-m_{new}} l_{partial} \cdot O_{partial}}{l_{new}}$$

总通信量：$P$ 步，每步传输一份 KV block。

## 1. 专家并行（Expert Parallelism）

MoE 模型中，每个 Transformer 层的 FFN 被替换为多个 Expert（小型 FFN），
Router 根据 token 内容选择 top-k 个 Expert 进行计算。

专家并行将这些 Expert 分布到不同 GPU 上：

```
  GPU 0        GPU 1        GPU 2        GPU 3
  ──────       ──────       ──────       ──────
  Expert 0     Expert 1     Expert 2     Expert 3
  Expert 4     Expert 5     Expert 6     Expert 7

  (8 个 Expert 分配到 4 张 GPU, 每张 2 个 Expert)
```

**Token Dispatch 流程**:
1. Router 在每个 GPU 上本地计算 token 的 top-k Expert
2. All-to-All 通信：将 token 发送到持有对应 Expert 的 GPU
3. 各 GPU 用本地 Expert 计算
4. All-to-All 通信：将结果送回原 GPU

In [ ]:
# 专家分配演示
print("=" * 50)
print("专家并行 (Expert Parallelism) 演示")
print("=" * 50)

# 均匀分配
n_experts = 8
world_size = 4
print(f"\n{n_experts} 个 Expert 分配到 {world_size} 张 GPU:")
for rank in range(world_size):
    experts = partition_experts(n_experts, rank, world_size)
    print(f"  Rank {rank}: Expert {experts}")

# Expert 归属查询
print(f"\nExpert 归属映射 (get_expert_owner):")
for expert_idx in range(n_experts):
    owner = get_expert_owner(expert_idx, world_size)
    print(f"  Expert {expert_idx} → Rank {owner}")

# 非均匀分配 (模拟 DeepSeek-V3 风格, 部分 Expert 为共享 Expert)
print(f"\n非均匀分配: {n_experts + 1} 个 Expert (含 1 个共享), {world_size} GPU:")
for rank in range(world_size):
    experts = partition_experts(n_experts + 1, rank, world_size)
    print(f"  Rank {rank}: Expert {experts}")

# EP 通信开销
print(f"\nEP 通信分析:")
print(f"  每层 2 次 All-to-All (token dispatch + result return)")
print(f"  All-to-All 复杂度: O(N) 总数据量, O(P-1/P) 步完成")
print(f"  通信量 = token 数量 × 隐藏维度 × top_k")

In [ ]:
# Token Dispatch 演示
print("\n" + "=" * 50)
print("Token Dispatch 演示")
print("=" * 50)

# 模拟: batch=2, seq=4, dim=8, 4 个 expert, top_k=2
B, S, D = 2, 4, 8
x = torch.randn(B, S, D)

# Router 输出 top-2 expert indices
router_indices = torch.tensor([
    [[0, 2], [1, 3], [0, 1], [2, 3]],
    [[1, 0], [3, 2], [2, 1], [0, 3]],
])

n_experts_demo = 4
dispatched = dispatch_tokens_to_experts(x, router_indices, n_experts_demo)

print(f"输入形状: {list(x.shape)}")
print(f"Router top-k: {router_indices.shape[-1]}")
print(f"总 Expert 数: {n_experts_demo}")
print(f"\nToken 分发结果:")
for expert_idx, tokens in dispatched.items():
    print(f"  Expert {expert_idx}: {tokens.shape[0]} 个 token, 形状 {list(tokens.shape)}")

# 负载均衡分析
print(f"\n负载均衡分析:")
token_counts = {e: dispatched[e].shape[0] if e in dispatched else 0 for e in range(n_experts_demo)}
for e, cnt in token_counts.items():
    bar = "█" * cnt
    print(f"  Expert {e}: {bar} ({cnt} tokens)")

total_tokens = sum(token_counts.values())
print(f"  总计: {total_tokens} token-dispatch 次")
print(f"  理想每 Expert: {total_tokens / n_experts_demo:.1f} tokens")
print(f"  ★ 负载不均衡是 MoE 训练的主要挑战之一")
print(f"  ★ DeepSeek-V3 使用辅助损失 (auxiliary loss) 来鼓励负载均衡")

### 📊 专家路由热力图

用热力图展示 token-expert 路由矩阵，直观观察负载均衡情况。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 模拟 token-expert 路由矩阵
np.random.seed(42)
n_tokens = 32
n_experts = 8
top_k = 2

# 生成路由概率
logits = np.random.randn(n_tokens, n_experts) * 2
# 让某些 expert 更受欢迎（模拟负载不均衡）
logits[:, 0] += 1.0  # Expert 0 更受欢迎
logits[:, 3] -= 0.5  # Expert 3 较少被选中

probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)

# 构建路由矩阵
routing = np.zeros((n_tokens, n_experts))
top_indices = np.argsort(logits, axis=1)[:, -top_k:]
for i in range(n_tokens):
    for j in top_indices[i]:
        routing[i, j] = probs[i, j]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左图: 路由热力图
im = axes[0].imshow(routing, cmap='YlOrRd', aspect='auto')
axes[0].set_xlabel('Expert 编号', fontsize=12)
axes[0].set_ylabel('Token 编号', fontsize=12)
axes[0].set_title('Token-Expert 路由热力图', fontsize=14)
plt.colorbar(im, ax=axes[0], label='路由概率')

# 右图: 每个 Expert 的负载
load = (routing > 0).sum(axis=0)
ideal = n_tokens * top_k / n_experts
colors = ['#e74c3c' if l > ideal * 1.3 else '#2ecc71' if l < ideal * 0.7 else '#3498db' for l in load]
axes[1].bar(range(n_experts), load, color=colors, edgecolor='black', linewidth=0.5)
axes[1].axhline(y=ideal, color='red', linestyle='--', linewidth=2, label=f'理想值={ideal:.0f}')
axes[1].set_xlabel('Expert 编号', fontsize=12)
axes[1].set_ylabel('被选中的 Token 数', fontsize=12)
axes[1].set_title('Expert 负载分布', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 2. 上下文并行（Context Parallelism）与 Ring Attention

当序列非常长（如 128K, 1M tokens）时，单张 GPU 无法容纳完整的注意力矩阵（O(S²) 显存）。
上下文并行通过沿序列维度切分注意力计算来解决这个问题。

**Ring Attention 核心思想**:
1. 将序列沿长度切分给各 GPU
2. 每个 GPU 计算本地 Q @ 本地 KV → partial attention
3. 将 KV block 沿环形拓扑传递给下一个 GPU
4. 重复直到 KV 绕行一圈
5. 在线 softmax 累积: 每步使用 running max 和 running sum 更新

```
  GPU 0: Q0 @ [K0, K1, K2, K3]   ← 环形传递 KV
  GPU 1: Q1 @ [K1, K2, K3, K0]
  GPU 2: Q2 @ [K2, K3, K0, K1]
  GPU 3: Q3 @ [K3, K0, K1, K2]

  通信量: O(N × P) 而非 O(N²) — 长序列场景的关键优势!
```

In [ ]:
# Ring Attention 概念演示
print("=" * 50)
print("Ring Attention 演示")
print("=" * 50)

B, n_heads, seq_len, d_head = 1, 2, 16, 4
q = torch.randn(B, n_heads, seq_len, d_head)
k = torch.randn(B, n_heads, seq_len, d_head)
v = torch.randn(B, n_heads, seq_len, d_head)

# 模拟环形传递 (4 步)
num_steps = 4
print(f"\n序列长度: {seq_len}, 每个 head 维度: {d_head}")
print(f"环步数: {num_steps} (每个 step 计算 1/4 的 KV)")

outputs = []
for step in range(num_steps):
    output = ring_attention_step(q, k, v, step)
    outputs.append(output)
    print(f"  Step {step}: output shape {list(output.shape)}")

# 显存对比
print(f"\n注意力显存对比:")
total_tokens = seq_len
attn_matrix_size = total_tokens * total_tokens  # 简化
print(f"  完整注意力:      O(S²) = {seq_len}² = {attn_matrix_size} 个元素")
print(f"  Ring Attention:   O(S/P) per step = {seq_len}/{num_steps} = {seq_len//num_steps} 个元素")
print(f"  显存节省:         ≈ {(1 - 1/num_steps)*100:.0f}% (注意力矩阵部分)")

# 通信分析
print(f"\nRing Attention 通信分析:")
print(f"  每步传输: 1 份 KV block (大小 = B × n_heads × (S/P) × d_head)")
print(f"  总通信量: P × KV_block = B × n_heads × S × d_head (与完整 KV 相同)")
print(f"  对比朴素 All-to-All: Ring 通信量不变，但延迟更可控")

### 📊 Ring Attention 分块示意图

可视化 Ring Attention 中 KV block 如何在环形拓扑中轮转。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

P = 4
colors_q = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
colors_kv = ['#85c1e9', '#82e0aa', '#f1948a', '#f9e79f']

for step in range(P):
    ax = axes[step]
    # 画环形
    angles = [90, 0, 270, 180]  # 上右下左
    for rank in range(P):
        angle_rad = np.radians(angles[rank])
        x = 2 * np.cos(angle_rad)
        y = 2 * np.sin(angle_rad)

        # Q block (本地)
        rect_q = mpatches.FancyBboxPatch((x-0.4, y+0.1), 0.8, 0.3,
                                          boxstyle='round,pad=0.02',
                                          facecolor=colors_q[rank], alpha=0.8)
        ax.add_patch(rect_q)
        ax.text(x, y+0.25, f'Q{rank}', ha='center', va='center', fontsize=8, fontweight='bold')

        # KV block (当前持有)
        kv_owner = (rank - step) % P
        rect_kv = mpatches.FancyBboxPatch((x-0.4, y-0.4), 0.8, 0.3,
                                           boxstyle='round,pad=0.02',
                                           facecolor=colors_kv[kv_owner], alpha=0.8)
        ax.add_patch(rect_kv)
        ax.text(x, y-0.25, f'KV{kv_owner}', ha='center', va='center', fontsize=8)

        # GPU label
        ax.text(x, y-0.7, f'GPU {rank}', ha='center', va='center', fontsize=9, fontweight='bold')

    # 画环形箭头
    for rank in range(P):
        next_rank = (rank + 1) % P
        a1 = np.radians(angles[rank])
        a2 = np.radians(angles[next_rank])
        ax.annotate('', xy=(1.5*np.cos(a2), 1.5*np.sin(a2)),
                    xytext=(1.5*np.cos(a1), 1.5*np.sin(a1)),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, connectionstyle='arc3,rad=0.2'))

    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_title(f'Step {step}', fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('Ring Attention: KV Block 环形轮转示意图', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. 上下文并行下的 Causal Mask

当序列被切分到多 GPU 后，因果注意力掩码需要相应调整：
- Rank 0: 标准下三角掩码（看不到任何其他 rank 的 token，因为是第一个分片）
- Rank k (>0): 需要看到前 k 个分片的所有 token + 自己分片内的 causal 约束

这被称为"扩展因果掩码"（extended causal mask），是实现正确性的关键。

In [ ]:
# CP Causal Mask 演示
print("=" * 50)
print("上下文并行 Causal Mask 演示")
print("=" * 50)

seq_len = 8
world_size = 2

print(f"\n序列长度: {seq_len}, CP size: {world_size}")
print(f"每个 rank 处理 {seq_len // world_size} 个 token")

for rank in range(world_size):
    mask = create_cp_causal_mask(seq_len=seq_len, rank=rank, world_size=world_size)
    chunk_size = seq_len // world_size
    local_start = rank * chunk_size
    
    print(f"\nRank {rank} causal mask (local tokens {local_start}:{local_start+chunk_size}):")
    print(f"  Mask 形状: {list(mask.shape)}")
    print(f"  可见 token 数 (False): {mask.numel() - mask.sum().item()}")
    print(f"  被掩盖 token 数 (True): {mask.sum().item()}")
    print(f"  可见范围: token 0 到 token {local_start + chunk_size - 1} (符合 causal)")

# 可视化: 哪个 rank 能看到哪些 token
print(f"\n序列切分可视化:")
for rank in range(world_size):
    chunk_size = seq_len // world_size
    local_start = rank * chunk_size
    visible = list(range(0, local_start + chunk_size))
    print(f"  Rank {rank} (持有 [{local_start}:{local_start+chunk_size})): 可看到 tokens {visible}")

# 不均分场景
print(f"\n非均匀切分 (seq_len=10, world_size=3):")
x_uneven = torch.randn(2, 10, 8)
for rank in range(3):
    chunk = partition_sequence(x_uneven, rank, 3)
    print(f"  Rank {rank}: chunk 形状 {list(chunk.shape)}")

## 4. CP + TP 混合策略显存分析

在大模型 + 长序列场景下，单独的 TP 或 CP 可能不够，需要组合使用。
TP 切分权重维度，CP 切分序列维度，两者互补：
- TP 减少每卡的模型权重显存
- CP 减少每卡的激活显存（序列维度）

总显存节省: ≈ 1 / (tp_size × cp_size)（理论值）

In [ ]:
# CP+TP 混合显存分析
print("=" * 50)
print("CP+TP 混合策略显存分析")
print("=" * 50)

configs = [
    (2048, 1024, 32, 2, 1, "标准 Llama 7B"),
    (4096, 2048, 32, 4, 2, "Llama 13B"),
    (8192, 4096, 64, 4, 4, "Llama 70B"),
    (16384, 4096, 64, 8, 4, "长序列大模型"),
    (32768, 5120, 128, 8, 8, "超长上下文"),
    (65536, 8192, 128, 8, 8, "128K 上下文"),
]

print(f"{'场景':<20s} {'seq':>6s} {'dim':>5s} {'TP':>3s} {'CP':>3s} {'单卡激活':>12s} {'节省比':>8s}")
print("-" * 70)

for label, seq_len, dim, n_heads, tp, cp in configs:
    mem = analyze_cp_tp_memory(seq_len, dim, n_heads, tp, cp)
    print(f"{label:<20s} {seq_len:>6d} {dim:>5d} {tp:>3d} {cp:>3d} "
          f"{mem['per_device_activation']:>10.1f}  {mem['reduction_ratio']:>7.1%}")

print(f"\n★ TP 切分模型权重, CP 切分激活序列 → 两者正交互补")
print(f"★ 总 GPU 数 = tp_size × cp_size × dp_size × pp_size")

# 并行策略推荐
print(f"\n" + "=" * 50)
print("并行策略推荐")
print("=" * 50)

scenarios = [
    (0.5, 2048, 4, "小模型 + 短序列"),
    (7.0, 4096, 8, "Llama 7B 风格"),
    (13.0, 8192, 16, "Llama 13B 风格"),
    (30.0, 16384, 32, "中等大模型 + 长序列"),
    (70.0, 32768, 64, "大模型 + 长序列"),
    (150.0, 65536, 128, "超大模型 + 超长序列"),
]

for model_gb, seq_len, n_gpus, desc in scenarios:
    config = recommend_parallel_config(model_gb, seq_len, n_gpus)
    print(f"  {desc:<25s}: {config}")

## 5. 关键要点总结

1. **专家并行 (EP)**: 将 MoE 的 Expert 分布到多 GPU，通过 All-to-All 通信实现 token dispatch。核心挑战是负载均衡。
2. **Token Dispatch**: Router 本地计算 → All-to-All 分发 token → 本地 Expert 计算 → All-to-All 回收结果
3. **上下文并行 (CP)**: 沿序列维度切分注意力计算，Ring Attention 将 O(S²) 显存降为 O(S/P)
4. **Ring Attention**: KV block 在环形拓扑中轮转，每步计算 partial attention + 在线 softmax 累积
5. **CP Causal Mask**: 需要扩展因果掩码，每个 rank 可见自己及之前所有分片的 token
6. **CP+TP 组合**: 两者正交互补 —— TP 减少权重显存，CP 减少激活显存
7. **推荐规则**: 模型 > 10GB 用 TP+PP+DP；序列 > 8K 加 CP；MoE 模型加 EP

## 📝 练习题

### 思考题
1. 在 Ring Attention 中，为什么需要在线 softmax 累积而不是简单的分步 softmax？如果直接对每步的 partial attention 输出求平均会怎样？

### 编程题
2. 实现一个简化的 Ring Attention：给定 4 个 GPU 各持有 4 个 token 的 Q/K/V，模拟 4 步环形传递，验证最终输出与完整注意力计算等价。

### 分析题
3. 一个 MoE 模型有 64 个路由专家（Top-4），使用 EP=8。计算：(a) 每个 GPU 分配多少专家？(b) 每层 All-to-All 通信量是多少（假设 dim=4096, seq_len=2048, batch=4）？(c) 如果负载极度不均衡（80% token 路由到 1 个 GPU），会出现什么问题？